# Notebook 03 — LSTM Trajectory Model
SkyGuard AI | Training the LSTM Trajectory Predictor

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../backend"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
print("Imports OK")

## 1. Load Training Data

In [ ]:
df = pd.read_csv("../backend/data/adsb_cleaned.csv")
df["time"] = pd.to_datetime(df["time"], errors="coerce")
df = df.sort_values(["icao24", "time"])
print("Rows:", len(df), "| Unique aircraft:", df.icao24.nunique())

## 2. Build Sequences
Window of 5 consecutive [lat, lon] points → predict next point.

In [ ]:
SEQ_LEN = 5
features = ["lat", "lon"]
X, y = [], []
for icao in df["icao24"].unique():
    vals = df[df["icao24"] == icao][features].dropna().values
    if len(vals) <= SEQ_LEN:
        continue
    for i in range(len(vals) - SEQ_LEN):
        X.append(vals[i:i+SEQ_LEN])
        y.append(vals[i+SEQ_LEN])
X, y = np.array(X), np.array(y)
print("Sequences shape:", X.shape, "| Targets shape:", y.shape)

## 3. Scale Data

In [ ]:
scaler = MinMaxScaler()
X_flat = X.reshape(-1, 2)
X_scaled = scaler.fit_transform(X_flat).reshape(X.shape)
y_scaled = scaler.transform(y)
print("Scaling done")

## 4. Build LSTM Model

In [ ]:
from keras.models import Sequential
from keras.layers import LSTM, Dense, Input

model = Sequential([
    Input(shape=(SEQ_LEN, 2)),
    LSTM(64, return_sequences=True),
    LSTM(32),
    Dense(16, activation="relu"),
    Dense(2)
])
model.compile(optimizer="adam", loss="mse")
model.summary()

## 5. Train

In [ ]:
history = model.fit(
    X_scaled, y_scaled,
    epochs=5,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

## 6. Plot Training Loss

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.title("LSTM Training Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Save Model and Scaler

In [ ]:
import joblib
model.save("../backend/data/lstm_model.keras")
joblib.dump(scaler, "../backend/data/scaler.pkl")
print("Saved lstm_model.keras + scaler.pkl")